# EMR Spark Connect Client Wrapper — supports all deployment models

`EMRSparkSession` connects a local PySpark process to a remote Spark driver over
[Spark Connect](https://spark.apache.org/docs/latest/spark-connect-overview.html) on
`EMR Serverless, EMR on EC2, and EMR on EKS`. The three flavors expose
quite different APIs; this wrapper hides that behind a single `create()` call and
keeps the connection's auth token fresh for as long as the notebook lives.

| | EMR Serverless | EMR on EC2 | EMR on EKS |
|---|---|---|---|
| `resource_id` | application ID<br>`00abcdef01234567` | cluster ID<br>`j-XXXXXXXXXXXXX` | virtual cluster ID<br>`m123456789abcdefg` |
| boto3 client | emr-serverless | emr | emr-containers |
| Remote resource | session | session | **managed endpoint** |
| Provisioned by | `StartSession` | `StartSession` | `CreateManagedEndpoint` |
| `execution_role_arn` | required | optional | required |
| `session.session_id` | session ID | session ID | managed endpoint ID |


### ⚠️ The PySpark version must match EMR Spark's major version



In [ ]:
# Run the followings via terminal from the REPO ROOT (one level up from examples/)
# to create two kernels. The kernels are registered globally with absolute paths,
# so this notebook works from examples/ (or anywhere) once they exist.
#
# Kernel A — EMR Serverless + EMR on EKS (Spark 3.5.x)
# python -m venv .venv-spark35
# ./.venv-spark35/bin/python -m pip install -e . "pyspark[connect]>=3.5.6,<4" "boto3>=1.43.72" ipykernel
# ./.venv-spark35/bin/python -m ipykernel install --user --name spark35 --display-name "EMR (Spark 3.5)"
#
# Kernel B — EMR on EC2 (Spark 4.0.2)
# python -m venv .venv-spark40
# ./.venv-spark40/bin/python -m pip install -e . "pyspark[connect]==4.0.2" "boto3>=1.43.72" ipykernel
# ./.venv-spark40/bin/python -m ipykernel install --user --name spark40 --display-name "EMR (Spark 4.0)"

#  ./.venv-spark35/bin/python -m jupyter kernelspec list


## 1. EMR on EC2 test on kernel EMR(Spark4.0)
Prerequisites:
- `a Spark-only cluster` with sessions enabled (emr-spark-8.0.0+, Spark 4.0.2)
- `execution_role_arn" is OPTIONAL` — supply it only for runtime-role sessions.
- The Spark Connect URL carries an extra `authorization={session_id}` parameter, which the wrapper adds for you.

In [ ]:
import os;os.environ["AWS_PROFILE"] = "default"
from emr_spark_connect import EMRSparkSession
session = EMRSparkSession.create(
        resource_id="j-1K48XXXXXXHCB",
        execution_role_arn="arn:aws:iam::123456789012:role/EMR_RuntimeRole_001",
        idle_timeout_minutes=1
    )
# validate the remote connection
session.range(5).selectExpr("id", "id * id AS squared").show()

### Test the idle timeout with EMR-EC2
- the remote session is gone — recreate() starts a new one.
- A new session is a new Spark driver: temp views and cached DataFrames are gone.

In [ ]:
if not session.is_active():
    old = session.session_id
    print(f"session {old} is {session.session_state()}")
    session.recreate()       # same settings, brand-new session
    print(f"recreated {old} -> {session.session_id}")

print(f"spark version  {session.spark.version}")
session.range(5).selectExpr("id", "id * id AS squared").show()

## 2. EMR Serverless test on another kernel - EMR(Spark 3.5)

Prerequisites:
- an application has interactive sessions enabled (EMR 7.13.0+, Spark 3.5.6+)
- a job execution role

In [ ]:
import os;os.environ["AWS_PROFILE"] = "default"
from emr_spark_connect import EMRSparkSession

serverless_session = EMRSparkSession.create(
    resource_id="00abcdef01234567",
    execution_role_arn="arn:aws:iam::123456789012:role/EMRServerlessS3RuntimeRole",
    idle_timeout_minutes=1
)
# validate the remote connection
serverless_session.range(5).selectExpr("id", "id * id AS squared").show()

### Test the idle timeout with EMR Serverless
- the remote session is gone — recreate() starts a new one.
- A new session is a new Spark driver: temp views and cached DataFrames are gone.

In [ ]:
if not serverless_session.is_active():
    old = serverless_session.session_id
    print(f"session {old} is {serverless_session.session_state()}")
    serverless_session.recreate()       # same settings, brand-new session
    print(f"recreated {old} -> {serverless_session.session_id}")
    
print(f"spark version  {serverless_session.spark.version}")
serverless_session.range(5).selectExpr("id", "id * id AS squared").show()

## 3. EMR on EKS
Prerequisites: 
- 1/ a Virtual Cluster (EMR 7.14.0+, Spark 3.5.8+) with session enabled
- 2/ a security configuration is attached to VC
- 3/ a job execution role
- 4/ an AWS Load Balancer Controller installed on the EKS cluster

EMR on EKS specific parameters: 
- managed_endpoint_id/release_label, token_duration_seconds, application_configuration, monitoring_configuration

In [ ]:
import os;os.environ["AWS_PROFILE"] = "default"
from emr_spark_connect import EMRSparkSession

eks_session = EMRSparkSession.create(
    resource_id="m123456789abcdefghijk",
    execution_role_arn="arn:aws:iam::123456789012:role/emr-on-eks-execution-role",
    idle_timeout_minutes=1,
    spark_conf={
        # avoid cross-AZ data traffic
        "spark.kubernetes.node.selector.topology.kubernetes.io/zone": "us-west-2a"
    },
    # Optional control-plane override (e.g. a beta endpoint):
    # endpoint_url="https://emr-containers-beta...",
)

# validate the remote connection
print(f"session id  {eks_session.session_id}")
print(f"spark version  {eks_session.spark.version}")
eks_session.range(5).selectExpr("id", "id * id AS squared").show()
old_endpoint = eks_session.session_id

### Which timer ran out: token or endpoint?

EMR on EKS is the only backend where "it stopped working" is ambiguous, because two things expire on **independent clocks**:

| Layer | `create()` argument | API name | Lifetime | Renewed by |
|---|---|---|---|---|
| Managed endpoint | `idle_timeout_minutes` | `sessionIdleTimeoutInMinutes` | max 24h | `CreateManagedEndpoint` — a *new* endpoint, on a *new* host |
| Session token | `token_duration_seconds` | `durationInSeconds` | max 12h | `GetManagedEndpointSessionCredentials` against the *same* endpoint |

Two calls tell them apart:

| Check | Cost | Meaning when True |
|---|---|---|
| `is_token_expired()` | local — compares the recorded `expiresAt` | token lapsed. **Nothing to do** — the gRPC interceptor remints on the next call, same endpoint, same Spark driver |
| `not is_endpoint_active()` | one `DescribeManagedEndpoint` call | endpoint hit its idle timeout — call `reconnect()` to attach to a replacement |

Note: with `token_duration_seconds` below 300 the token is always inside the interceptor's five-minute early-refresh window, so it is reminted on every single gRPC call — which is why a lapsed token is hard to observe in practice.

In [ ]:
from emr_spark_connect import EndpointExpiredError

if not eks_session.is_endpoint_active():
     print("Managed endpoint idled out, reconnecting to a replacement ...")                                              
     eks_session.reconnect()
     print(f"endpoint replaced {old_endpoint} -> {eks_session.session_id}")
if eks_session.is_token_expired():
     print(f"token     duration = {eks_session.token_duration_seconds}s  "
       f"remaining = {eks_session.token_seconds_remaining():.0f}s  "
       f"expired = {eks_session.is_token_expired()}")
     print("\nToken lapsed, the interceptor remints it on the next call ...")
try:
    eks_session.range(5).selectExpr("id", "id * id AS squared").show()
except (EndpointExpiredError) as e:
    print("\nManaged endpoint idled out, reconnecting to a replacement ...")
    eks_session.reconnect()
    print(f"endpoint replaced {old_endpoint} -> {eks_session.session_id}")
    # A new Spark driver
    eks_session.range(5).selectExpr("id", "id * id AS squared").show()

# stop Spark session, keep the endpoint alive until idle timeout
active_endpoint = eks_session.session_id
# eks_session.stop(terminate=False)


### Reconnect with a managed endpoint id

| Scenario | Action |
|---|---|
| Pass in an `ACTIVE` endpoint (`managed_endpoint_id` set) | reuse |
| Pass in an invalid endpoint (`EXPIRED` or `TERMINATED`) | create a new |
| `managed_endpoint_id` = None (default) | create a new |

In [ ]:
from emr_spark_connect import EMRSparkSession
reuse_session = EMRSparkSession.create(
    resource_id="m123456789abcdefghijk",
    execution_role_arn="arn:aws:iam::123456789012:role/emr-on-eks-execution-role",
    # Active EP
    managed_endpoint_id=active_endpoint,
)
# validate the remote connection
print(f"session_id     {reuse_session.session_id}")
reuse_session.range(5).selectExpr("id", "id * id AS squared").show()

### Endpoint expired: pass in an expired endpoint id


In [ ]:
expired_session = EMRSparkSession.create(
    resource_id="m123456789abcdefghijk",
    execution_role_arn="arn:aws:iam::123456789012:role/emr-on-eks-execution-role",
    idle_timeout_minutes=1,
    spark_conf={
        "spark.dynamicAllocation.minExecutors": "0",
        "spark.dynamicAllocation.enabled": "true"
    },
    # expired endpoint, create a new
    managed_endpoint_id="xxxxxxxxxxxxx",
)
print(f"session_id     {expired_session.session_id}")
print(f"spark version  {expired_session.spark.version}")
expired_session.range(5).selectExpr("id", "id * id AS squared").show()

# stop Spark session
# terminate the managed endpoint before its idle timeout
expired_session.stop(terminate=True)